# Synthetic Data Experiment Series
### All projmix cases
- Data generator: Gaussian/Projected-Gaussian
- Data dimension: 2d / 3d
- Separability between clusters: high / low
- Cross-correlation: strong / weak / independent


## 01 - Data Generation

In [1]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path
from time import time
import numpy as np
import pandas as pd

import cyl_lvm as clvm
from experiment.synthetic_data import experiment_helper as mod

CONFIG_PATH = Path("projmix_configs.json")

with CONFIG_PATH.open("r") as f:
    generator_configs = json.load(f)["experiments"]

In [2]:
N = 10000
N_train = int(N * 0.8)
NOISE_SCALE = 0.15
OUTPUT_PATH_AGG = Path("projmix_results_raw.csv")
OUTPUT_PATH_LONG = Path("projmix_results_raw_long.csv")

setup_list = [
    {"model_type": "cylmix", "model_components": [3]},
    {"model_type": "indcylmix", "model_components": [3]},
    {"model_type": "mom", "model_components": [3, 1]},
    {"model_type": "mom", "model_components": [3, 2]},
    {"model_type": "isomom", "model_components": [3, 1]},
    {"model_type": "isomom", "model_components": [3, 2]},

    {"model_type": "cylmix", "model_components": [4]},
    {"model_type": "indcylmix", "model_components": [4]},
    {"model_type": "mom", "model_components": [4, 1]},
    {"model_type": "mom", "model_components": [4, 2]},
    {"model_type": "isomom", "model_components": [4, 1]},
    {"model_type": "isomom", "model_components": [4, 2]},

    {"model_type": "cylmix", "model_components": [2]},
    {"model_type": "indcylmix", "model_components": [2]},
    {"model_type": "mom", "model_components": [2, 1]},
    {"model_type": "mom", "model_components": [2, 2]},
    {"model_type": "isomom", "model_components": [2, 1]},
    {"model_type": "isomom", "model_components": [2, 2]},
]

model_names = [
    "3-Cylindrical Mixture",
    "3-Independent Cylindrical Mixture",
    "(3,1)-Two-layer MoM",
    "(3,2)-Two-layer MoM",
    "(3,1)-Isolated Two-layer MoM",
    "(3,2)-Isolated Two-layer MoM",

    "4-Cylindrical Mixture",
    "4-Independent Cylindrical Mixture",
    "(4,1)-Two-layer MoM",
    "(4,2)-Two-layer MoM",
    "(4,1)-Isolated Two-layer MoM",
    "(4,2)-Isolated Two-layer MoM",

    "2-Cylindrical Mixture",
    "2-Independent Cylindrical Mixture",
    "(2,1)-Two-layer MoM",
    "(2,2)-Two-layer MoM",
    "(2,1)-Isolated Two-layer MoM",
    "(2,2)-Isolated Two-layer MoM",
]

experiment_order = [key.replace("projmix-", "", 1) for key in generator_configs.keys()]


In [3]:
records = []
n_seeds = 100
seeds = range(n_seeds)
total_progress = len(generator_configs) * len(seeds)
progress_idx = 0

for key, cfg in generator_configs.items():
    case = key.replace("projmix-", "", 1)
    case_start = time()

    d_gauss = cfg["dimensions"]["euclidean"]
    d_vmf = cfg["dimensions"]["projected"]
    d_total = cfg["dimensions"]["total"]

    component_means = np.asarray(cfg["component_means"], dtype=float)
    covariances = np.asarray(cfg["covariances"], dtype=float)
    weights = np.asarray(cfg["mixture_weights"], dtype=float)

    components = [
        clvm.MultivariateGaussian(
            d_total,
            mean=mean,
            covariance=covariance,
        )
        for mean, covariance in zip(component_means, covariances)
    ]

    for seed in seeds:
        seed_start = time()
        rng = np.random.RandomState(seed)

        generator = clvm.MixtureModel(
            components=components,
            weights=weights,
            init="k-means",
            rng=rng,
        )

        sample = mod.sample_noisy_train_test(
            generator,
            n=N,
            n_train=N_train,
            d_gauss=d_gauss,
            d_vmf=d_vmf,
            rng=rng,
            noise_scale=NOISE_SCALE,
        )

        labels_train = sample["labels_train"]
        labels_test = sample["labels_test"]

        models, training_times, n_iters = mod.train_all_models(
            d_gauss,
            sample["x_train"],
            setup_list=setup_list,
            print_=False,
            return_training_times=True,
            return_em_iter=True,
        )
        models_with_noise, training_times_with_noise, n_iters_with_noise = mod.train_all_models(
            d_gauss,
            sample["x_train_noise"],
            setup_list=setup_list,
            print_=False,
            return_training_times=True,
            return_em_iter=True,
        )

        runs = {
            "no noise": {
                "models": models,
                "training_times": training_times,
                "em_iters": n_iters,
                "x_train": sample["x_train"],
                "x_test": sample["x_test"],
            },
            "with noise": {
                "models": models_with_noise,
                "training_times": training_times_with_noise,
                "em_iters": n_iters_with_noise,
                "x_train": sample["x_train_noise"],
                "x_test": sample["x_test_noise"],
            },
        }

        for noise, run in runs.items():
            records.append({
                    "Seed": seed,
                    "Model": "all",
                    "Model Type": "all",
                    "Metric": "sample_cross_cov",
                    "Sample": "training",
                    "Noise": noise,
                    "Experiment": case,
                    "Value": round(np.linalg.norm(np.cov(run["x_train"],
                                                         rowvar=False)[:d_gauss,d_gauss:],
                                           ord='fro'), 2),
                })
            for model_name, setup, elapsed, em_iters in zip(
                model_names,
                setup_list,
                run["training_times"],
                run["em_iters"],
            ):
                records.append({
                    "Seed": seed,
                    "Model": model_name,
                    "Model Type": setup["model_type"],
                    "Metric": "training_time",
                    "Sample": "training",
                    "Noise": noise,
                    "Experiment": case,
                    "Value": round(elapsed*1000, 2),
                })
                records.append({
                    "Seed": seed,
                    "Model": model_name,
                    "Model Type": setup["model_type"],
                    "Metric": "em_iters",
                    "Sample": "training",
                    "Noise": noise,
                    "Experiment": case,
                    "Value": int(em_iters),
                })

            for sample_name, x_eval, labels in [
                ("in sample", run["x_train"], labels_train),
                ("out of sample", run["x_test"], labels_test),
            ]:
                for model, model_name, setup in zip(run["models"], model_names, setup_list):
                    records.append({
                        "Seed": seed,
                        "Model": model_name,
                        "Model Type": setup["model_type"],
                        "Metric": "ari",
                        "Sample": sample_name,
                        "Noise": noise,
                        "Experiment": case,
                        "Value": round(mod.ari_model(model,
                                                     labels,
                                                     x_eval,
                                                     d_gauss), 5),
                    })


                    for score in ["gmpd", "avg_ll", "bic", "aic"]:
                        records.append({
                        "Seed": seed,
                        "Model": model_name,
                        "Model Type": setup["model_type"],
                        "Metric": score,
                        "Sample": sample_name,
                        "Noise": noise,
                        "Experiment": case,
                        "Value": round(mod.score_model(model,
                                                       x_eval,
                                                       d_gauss,
                                                       score_type=score), 5),
                        })


        progress_idx += 1
        print(
            f"{progress_idx/total_progress*100:.2f}% | "
            f"{case} | seed={seed} | "
            f"seed time: {time() - seed_start:.0f}s | "
            f"case elapsed: {time() - case_start:.0f}s",
            flush=True,
        )

    print(
        f"{progress_idx}/{total_progress} | "
        f"{case} complete | total case time: {time() - case_start:.0f}s",
        flush=True,
    )

results_long = pd.DataFrame(records)

0.08% | 2d-low-strong | seed=0 | seed time: 34s | case elapsed: 34s
0.17% | 2d-low-strong | seed=1 | seed time: 29s | case elapsed: 64s
0.25% | 2d-low-strong | seed=2 | seed time: 37s | case elapsed: 100s
0.33% | 2d-low-strong | seed=3 | seed time: 29s | case elapsed: 129s
0.42% | 2d-low-strong | seed=4 | seed time: 33s | case elapsed: 162s
0.50% | 2d-low-strong | seed=5 | seed time: 27s | case elapsed: 189s
0.58% | 2d-low-strong | seed=6 | seed time: 27s | case elapsed: 216s
0.67% | 2d-low-strong | seed=7 | seed time: 25s | case elapsed: 241s
0.75% | 2d-low-strong | seed=8 | seed time: 29s | case elapsed: 270s
0.83% | 2d-low-strong | seed=9 | seed time: 25s | case elapsed: 295s
0.92% | 2d-low-strong | seed=10 | seed time: 27s | case elapsed: 321s
1.00% | 2d-low-strong | seed=11 | seed time: 31s | case elapsed: 352s
1.08% | 2d-low-strong | seed=12 | seed time: 26s | case elapsed: 378s
1.17% | 2d-low-strong | seed=13 | seed time: 30s | case elapsed: 409s
1.25% | 2d-low-strong | seed=14 

In [4]:
experiment_parts = (
    results_long["Experiment"]
    .str.replace(r"^projmix-", "", regex=True)
    .str.extract(
        r"^(?P<dimension>\d+)d-(?P<separability>[^-]+)-(?P<dependence>[^-]+)$"
    )
)

results_long = results_long.assign(
    Dimension=experiment_parts["dimension"].astype(int),
    Separability=experiment_parts["separability"],
    Dependence=experiment_parts["dependence"],
)

In [5]:
results_long[["K", "L"]] = results_long["Model"].str.extract(
    r"^\(?(\d+)(?:,(\d+))?"
)
results_long["K"] = results_long["K"].fillna(0).astype(int)
results_long["L"] = results_long["L"].fillna(0).astype(int)

In [6]:
results_long.columns

Index(['Seed', 'Model', 'Model Type', 'Metric', 'Sample', 'Noise',
       'Experiment', 'Value', 'Dimension', 'Separability', 'Dependence', 'K',
       'L'],
      dtype='object')

In [7]:
agg_cols = [
    "Model",
    "Model Type",
    "Metric",
    "Sample",
    "Noise",
    "Experiment",
    "Dimension",
    "Separability",
    "Dependence",
    "K",
    "L"
]

results_agg = (
    results_long
    .groupby(agg_cols, as_index=False)
    .agg(
        **{
            "Avg Value": ("Value", "mean"),
            "Std Value": ("Value", "std"),
        }
    )
)

results_agg["Std Value"] = results_agg["Std Value"].fillna(0.0)

In [8]:
results_agg.to_csv(
    OUTPUT_PATH_AGG,
    index=False
)

In [9]:
results_long.to_csv(
    OUTPUT_PATH_LONG,
    index=False
)